# SAMPLE PRE-PROCESSING

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm, AnovaRM
from statsmodels.multivariate.manova import MANOVA

In [2]:
df = pd.read_csv(
    "https://huggingface.co/datasets/behAIvNET/spemetryxyz/resolve/main/spemetryxyz.csv"
)

In [3]:
data = df.rename(columns={
    "Student ID": "ID",
    "Official Grade": "Grade",
    "Initial Grade": "Level",
    "Baseline RR": "RR0",
    "Baseline RA": "RA0",
    "Baseline RC": "RC0",
    "Intervention Approach": "Approach",
    "Intervention Intensity": "Intensity",
})

severity_order = ["Very Mild", "Mild", "Moderate", "Severe", "Very Severe"]
data["Severity"] = pd.Categorical(data["Severity"], categories=severity_order, ordered=True)

alpha = 0.05
power = 0.80
rng = np.random.default_rng(42)

resample_data = data.sample(n=2000, random_state=42).reset_index(drop=True)
distance_data = data.sample(n=300, random_state=42).reset_index(drop=True)
n_permutation = 1000

print(data.shape)
print(data.dtypes.head(9))

(100000, 39)
ID              int64
Grade           int64
RR0             int64
RA0             int64
RC0             int64
Level           int64
Severity     category
Approach       object
Intensity       int64
dtype: object


In [4]:
def interpret(value, thresholds, labels):
    return labels[int(np.searchsorted(thresholds, abs(value), side="right"))]

def decide(p_value):
    return "reject H0" if p_value < alpha else "fail to reject H0"


## SAMPLE SIZE

In [5]:
z_alpha = stats.norm.ppf(1 - alpha / 2)
z_beta = stats.norm.ppf(power)

group_1 = data.loc[data["Approach"] == "Phonics", "W10RR"]
group_2 = data.loc[data["Approach"] == "Whole-Word", "W10RR"]
sigma = np.sqrt((group_1.var(ddof=1) + group_2.var(ddof=1)) / 2)
delta = abs(group_1.mean() - group_2.mean())

difference = data["W10RR"] - data["W01RR"]
sigma_difference = difference.std(ddof=1)
delta_paired = abs(difference.mean())

grand_mean = data["W10RR"].mean()
between = sum(len(g) * (g["W10RR"].mean() - grand_mean) ** 2
              for _, g in data.groupby("Severity", observed=True)) / len(data)
cohen_f = np.sqrt(between / data["W10RR"].var(ddof=1))

table = pd.crosstab(data["Approach"], data["Severity"])
chi_square = stats.chi2_contingency(table).statistic
cohen_w = np.sqrt(chi_square / table.values.sum())

r = data["RR0"].corr(data["RC0"])
fisher_z = 0.5 * np.log((1 + r) / (1 - r))

u = 3
v = 3

requirement = pd.DataFrame({
    "Test Family": [
        "Independent t & Wilcoxon Rank-Sum",
        "Paired t & Wilcoxon Signed-Rank",
        "One-Way ANOVA, RM ANOVA, Kruskal-Wallis, Friedman",
        "Two-Way ANOVA, SRH, ART, Chi-Square",
        "MANOVA, PERMANOVA, Wilks Lambda",
        "Pearson & Spearman Correlation",
    ],
    "Effect Size": [delta, delta_paired, cohen_f, cohen_w, cohen_f, r],
    "Required n": [
        2 * ((z_alpha + z_beta) * sigma / delta) ** 2,
        ((z_alpha + z_beta) * sigma_difference / delta_paired) ** 2,
        (z_alpha + z_beta) ** 2 / cohen_f ** 2,
        (z_alpha + z_beta) ** 2 / cohen_w ** 2,
        (z_alpha + z_beta) ** 2 * (u + v + 1) / (u * cohen_f ** 2),
        (z_alpha + z_beta) ** 2 / fisher_z ** 2,
    ],
})
requirement["Required n"] = np.ceil(requirement["Required n"])
requirement["Available n"] = len(data)
requirement["Sufficient"] = requirement["Available n"] >= requirement["Required n"]

print("Z(alpha):", round(z_alpha, 4), "| Z(beta):", round(z_beta, 4))
print(requirement.round(4).to_string(index=False))

Z(alpha): 1.96 | Z(beta): 0.8416
                                      Test Family  Effect Size  Required n  Available n  Sufficient
                Independent t & Wilcoxon Rank-Sum       7.1912       523.0       100000        True
                  Paired t & Wilcoxon Signed-Rank      27.0121         1.0       100000        True
One-Way ANOVA, RM ANOVA, Kruskal-Wallis, Friedman       0.1568       320.0       100000        True
              Two-Way ANOVA, SRH, ART, Chi-Square       0.4706        36.0       100000        True
                  MANOVA, PERMANOVA, Wilks Lambda       0.1568       746.0       100000        True
                   Pearson & Spearman Correlation       0.9925         2.0       100000        True


## SAMPLE DISTRIBUTION

In [6]:
deviation_bands = [(0.95, "none"), (0.90, "slight"), (0.80, "moderate"), (0.00, "strong")]

normality = []
for column in ["RR0", "RA0", "RC0", "W10RR"]:
    values = data[column].sample(n=5000, random_state=42)
    w_stat, p_value = stats.shapiro(values)
    normality.append({
        "Variable": column,
        "W": round(w_stat, 4),
        "p-value": p_value,
        "Deviation": next(label for edge, label in deviation_bands if w_stat >= edge),
        "Decision": decide(p_value),
        "Route": "parametric" if p_value >= alpha else "non-parametric",
    })

print(pd.DataFrame(normality).to_string(index=False))

Variable      W      p-value Deviation  Decision          Route
     RR0 0.9541 5.300438e-37      none reject H0 non-parametric
     RA0 0.9580 1.005871e-35      none reject H0 non-parametric
     RC0 0.9604 6.662508e-35      none reject H0 non-parametric
   W10RR 0.9646 2.392037e-33      none reject H0 non-parametric


# CONFIRMATORY DATA ANALYSIS

## INTER-GROUP ANALYSIS

### Parametric Tests For Continuous Numerical Inter-Group

#### Independent t-Test

In [7]:
group_1 = data.loc[data["Approach"] == "Phonics", "W10RR"]
group_2 = data.loc[data["Approach"] == "Whole-Word", "W10RR"]

t_stat, p_value = stats.ttest_ind(group_1, group_2, equal_var=False)
degrees = len(group_1) + len(group_2) - 2
p_manual = 2 * (1 - stats.t.cdf(abs(t_stat), degrees))

print("Phonics:", len(group_1), "| mean:", round(group_1.mean(), 3))
print("Whole-Word:", len(group_2), "| mean:", round(group_2.mean(), 3))
print("t:", round(t_stat, 4), "| df:", degrees)
print("Difference:", interpret(t_stat, [1, 2, 3, 5],
                               ["no", "weak", "moderate", "strong", "very strong"]))
print("p-value (two-tailed):", p_value)
print("p-value (CDF):", p_manual)
print("Decision:", decide(p_value))

Phonics: 31697 | mean: 156.923
Whole-Word: 31530 | mean: 149.731
t: 21.7976 | df: 63225
Difference: very strong
p-value (two-tailed): 5.969852231923757e-105
p-value (CDF): 0.0
Decision: reject H0


#### Paired t-Test

In [8]:
before = data["W01RR"]
after = data["W10RR"]

t_stat, p_value = stats.ttest_rel(after, before)
degrees = len(before) - 1
p_manual = 2 * (1 - stats.t.cdf(abs(t_stat), degrees))

print("Pairs:", len(before))
print("Mean before:", round(before.mean(), 3), "| Mean after:", round(after.mean(), 3))
print("Mean difference:", round((after - before).mean(), 3))
print("t:", round(t_stat, 4), "| df:", degrees)
print("Difference:", interpret(t_stat, [1, 2, 3, 5],
                               ["no", "weak", "moderate", "strong", "very strong"]))
print("p-value (two-tailed):", p_value)
print("p-value (CDF):", p_manual)
print("Decision:", decide(p_value))

Pairs: 100000
Mean before: 126.301 | Mean after: 153.313
Mean difference: 27.012
t: 1323.7916 | df: 99999
Difference: very strong
p-value (two-tailed): 0.0
p-value (CDF): 0.0
Decision: reject H0


#### One-Way ANOVA Test

In [9]:
model = ols("W10RR ~ C(Severity)", data=data).fit()
table = anova_lm(model, typ=2)
print(table)

f_stat = table.loc["C(Severity)", "F"]
df_effect = table.loc["C(Severity)", "df"]
df_residual = table.loc["Residual", "df"]
p_value = 1 - stats.f.cdf(f_stat, df_effect, df_residual)

print(data.groupby("Severity", observed=True)["W10RR"].mean().round(3))
print("F:", round(f_stat, 4), "| df:", int(df_effect), int(df_residual))
print("p-value (CDF):", p_value)
print("Decision:", decide(p_value))

                   sum_sq       df           F  PR(>F)
C(Severity)  4.268308e+06      4.0  629.892608     0.0
Residual     1.693977e+08  99995.0         NaN     NaN
Severity
Very Mild      171.330
Mild           163.395
Moderate       153.366
Severe         143.179
Very Severe    133.846
Name: W10RR, dtype: float64
F: 629.8926 | df: 4 99995
p-value (CDF): 1.1102230246251565e-16
Decision: reject H0


#### Two-Way ANOVA Test

In [10]:
model = ols("W10RR ~ C(Severity) * C(Approach)", data=data).fit()
table = anova_lm(model, typ=2)
print(table)

df_residual = table.loc["Residual", "df"]
for effect in ["C(Severity)", "C(Approach)", "C(Severity):C(Approach)"]:
    f_stat = table.loc[effect, "F"]
    df_effect = table.loc[effect, "df"]
    p_value = 1 - stats.f.cdf(f_stat, df_effect, df_residual)
    print(effect, "| F:", round(f_stat, 4), "| df:", int(df_effect), int(df_residual),
          "| p-value:", p_value, "|", decide(p_value))

                               sum_sq       df             F    PR(>F)
C(Severity)              1.116649e+02      4.0  1.647849e-02  0.983657
C(Approach)              3.054998e-08      2.0  9.016572e-12  1.000000
C(Severity):C(Approach)  5.061509e+02      8.0  3.734656e-02  0.963342
Residual                 1.693949e+08  99991.0           NaN       NaN
C(Severity) | F: 0.0165 | df: 4 99991 | p-value: 0.9994686947534164 | fail to reject H0
C(Approach) | F: 0.0 | df: 2 99991 | p-value: 0.9999999999909834 | fail to reject H0
C(Severity):C(Approach) | F: 0.0373 | df: 8 99991 | p-value: 0.999981579253878 | fail to reject H0


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 4, but rank is 2
  warnings.warn('covariance of constraints does not have full '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 8, but rank is 2
  warnings.warn('covariance of constraints does not have full '


#### One-Way Repeated Measures ANOVA Test

In [11]:
subjects = data.sample(n=300, random_state=42)
long = subjects.melt(id_vars="ID", value_vars=["W01RR", "W05RR", "W10RR"],
                     var_name="Week", value_name="RR")

result = AnovaRM(long, depvar="RR", subject="ID", within=["Week"]).fit()
print(result)

table = result.anova_table
f_stat = table.loc["Week", "F Value"]
p_value = 1 - stats.f.cdf(f_stat, table.loc["Week", "Num DF"], table.loc["Week", "Den DF"])

print(long.groupby("Week")["RR"].mean().round(3))
print("F:", round(f_stat, 4), "| df:", table.loc["Week", "Num DF"], table.loc["Week", "Den DF"])
print("p-value (CDF):", p_value)
print("Decision:", decide(p_value))

                Anova
      F Value  Num DF  Den DF  Pr > F
-------------------------------------
Week 5971.9433 2.0000 598.0000 0.0000

Week
W01RR    124.740
W05RR    136.830
W10RR    151.997
Name: RR, dtype: float64
F: 5971.9433 | df: 2.0 598.0
p-value (CDF): 1.1102230246251565e-16
Decision: reject H0


#### Two-Way Repeated Measures ANOVA Test

In [12]:
subjects = data.sample(n=300, random_state=42)
frames = []
for week in ["W01", "W05", "W10"]:
    for measure in ["RR", "RA"]:
        frames.append(pd.DataFrame({
            "ID": subjects["ID"].values,
            "Week": week,
            "Measure": measure,
            "Score": subjects[week + measure].values,
        }))
long = pd.concat(frames, ignore_index=True)

result = AnovaRM(long, depvar="Score", subject="ID", within=["Week", "Measure"]).fit()
print(result)

table = result.anova_table
for effect in table.index:
    f_stat = table.loc[effect, "F Value"]
    p_value = 1 - stats.f.cdf(f_stat, table.loc[effect, "Num DF"], table.loc[effect, "Den DF"])
    print(effect, "| F:", round(f_stat, 4), "| p-value:", p_value, "|", decide(p_value))

                    Anova
              F Value  Num DF  Den DF  Pr > F
---------------------------------------------
Week         6024.5853 2.0000 598.0000 0.0000
Measure       622.8970 1.0000 299.0000 0.0000
Week:Measure    0.1078 2.0000 598.0000 0.8978

Week | F: 6024.5853 | p-value: 1.1102230246251565e-16 | reject H0
Measure | F: 622.897 | p-value: 1.1102230246251565e-16 | reject H0
Week:Measure | F: 0.1078 | p-value: 0.8978163391329441 | fail to reject H0


#### One-Way MANOVA Test

In [13]:
model = MANOVA.from_formula("W10RR + W10RA + W10RC ~ C(Severity)", data=data)
result = model.mv_test()
print(result)

table = result.results["C(Severity)"]["stat"]
wilks = table.loc["Wilks' lambda", "Value"]
f_stat = table.loc["Wilks' lambda", "F Value"]
p_value = table.loc["Wilks' lambda", "Pr > F"]

print("Wilks Lambda:", round(wilks, 6))
print("F:", round(f_stat, 4), "| df:", table.loc["Wilks' lambda", "Num DF"],
      table.loc["Wilks' lambda", "Den DF"])
print("p-value:", p_value)
print("Decision:", decide(p_value))

                     Multivariate linear model
                                                                   
-------------------------------------------------------------------
        Intercept        Value  Num DF   Den DF    F Value   Pr > F
-------------------------------------------------------------------
           Wilks' lambda 0.7312 3.0000 99993.0000 12252.2125 0.0000
          Pillai's trace 0.2688 3.0000 99993.0000 12252.2125 0.0000
  Hotelling-Lawley trace 0.3676 3.0000 99993.0000 12252.2125 0.0000
     Roy's greatest root 0.3676 3.0000 99993.0000 12252.2125 0.0000
-------------------------------------------------------------------
                                                                   
-------------------------------------------------------------------
      C(Severity)       Value   Num DF    Den DF    F Value  Pr > F
-------------------------------------------------------------------
          Wilks' lambda 0.9603 12.0000 264556.9023  340.1101 0.0000
 

#### Two-Way MANOVA Test

In [14]:
counts = pd.crosstab(data["Severity"], data["Approach"])
print(counts)

approach_levels = ["Multi-Sensory", "Phonics"]
severity_levels = counts.index[(counts[approach_levels] > 0).all(axis=1)]

two_way = data[data["Approach"].isin(approach_levels)
               & data["Severity"].isin(severity_levels)].copy()
two_way["Severity"] = two_way["Severity"].cat.remove_unused_categories()
print(pd.crosstab(two_way["Severity"], two_way["Approach"]))

model = MANOVA.from_formula("W10RR + W10RA + W10RC ~ C(Severity) * C(Approach)", data=two_way)
result = model.mv_test()

for effect in ["C(Severity)", "C(Approach)", "C(Severity):C(Approach)"]:
    table = result.results[effect]["stat"]
    wilks = table.loc["Wilks' lambda", "Value"]
    f_stat = table.loc["Wilks' lambda", "F Value"]
    p_value = table.loc["Wilks' lambda", "Pr > F"]
    print(effect, "| Wilks Lambda:", round(wilks, 6), "| F:", round(f_stat, 4),
          "| p-value:", p_value, "|", decide(p_value))

Approach     Multi-Sensory  Phonics  Whole-Word
Severity                                       
Very Mild                0     2000           0
Mild                  6940     7060           0
Moderate             22769    22637       22594
Severe                7064        0        6936
Very Severe              0        0        2000
Approach  Multi-Sensory  Phonics
Severity                        
Mild               6940     7060
Moderate          22769    22637
C(Severity) | Wilks Lambda: 0.991771 | F: 164.2836 | p-value: 4.653029621544664e-106 | reject H0
C(Approach) | Wilks Lambda: 0.999978 | F: 0.4425 | p-value: 0.7225883611839907 | fail to reject H0
C(Severity):C(Approach) | Wilks Lambda: 0.999987 | F: 0.2664 | p-value: 0.8496846438540104 | fail to reject H0


#### One-Way Repeated Measures MANOVA Test

In [15]:
subjects = data.sample(n=1000, random_state=42)
differences = pd.DataFrame({
    "dRR": subjects["W10RR"].values - subjects["W01RR"].values,
    "dRA": subjects["W10RA"].values - subjects["W01RA"].values,
    "dRC": subjects["W10RC"].values - subjects["W01RC"].values,
})

n = len(differences)
p = differences.shape[1]
mean_vector = differences.mean().values
covariance = np.cov(differences.values, rowvar=False)
t_squared = n * mean_vector @ np.linalg.inv(covariance) @ mean_vector
f_stat = t_squared * (n - p) / (p * (n - 1))
p_value = 1 - stats.f.cdf(f_stat, p, n - p)
wilks = 1 / (1 + t_squared / (n - 1))

print(differences.mean().round(3))
print("Hotelling T-squared:", round(t_squared, 4))
print("Wilks Lambda:", round(wilks, 6))
print("F:", round(f_stat, 4), "| df:", p, n - p)
print("p-value (CDF):", p_value)
print("Decision:", decide(p_value))

dRR    26.882
dRA    26.892
dRC    26.932
dtype: float64
Hotelling T-squared: 17995.1546
Wilks Lambda: 0.052595
F: 5986.3761 | df: 3 997
p-value (CDF): 1.1102230246251565e-16
Decision: reject H0


#### Two-Way Repeated Measures MANOVA Test

In [16]:
subjects = two_way.sample(n=1000, random_state=42).reset_index(drop=True)
paired = pd.DataFrame({
    "dRR": subjects["W10RR"] - subjects["W01RR"],
    "dRA": subjects["W10RA"] - subjects["W01RA"],
    "dRC": subjects["W10RC"] - subjects["W01RC"],
    "Approach": subjects["Approach"],
    "Severity": subjects["Severity"],
})
print(pd.crosstab(paired["Severity"], paired["Approach"]))

model = MANOVA.from_formula("dRR + dRA + dRC ~ C(Approach) * C(Severity)", data=paired)
result = model.mv_test()

for effect in ["C(Approach)", "C(Severity)", "C(Approach):C(Severity)"]:
    table = result.results[effect]["stat"]
    wilks = table.loc["Wilks' lambda", "Value"]
    f_stat = table.loc["Wilks' lambda", "F Value"]
    p_value = table.loc["Wilks' lambda", "Pr > F"]
    print(effect, "| Wilks Lambda:", round(wilks, 6), "| F:", round(f_stat, 4),
          "| p-value:", p_value, "|", decide(p_value))

Approach  Multi-Sensory  Phonics
Severity                        
Mild                116      118
Moderate            369      397
C(Approach) | Wilks Lambda: 0.996482 | F: 1.1699 | p-value: 0.3200872632025442 | fail to reject H0
C(Severity) | Wilks Lambda: 0.214078 | F: 1216.3924 | p-value: 0.0 | reject H0
C(Approach):C(Severity) | Wilks Lambda: 0.996391 | F: 1.1999 | p-value: 0.30862968095438564 | fail to reject H0


### Non-Parametric Tests For Numerical & Ordinal Categorical Inter-Group

#### Wilcoxon Rank-Sum Test For Independent t-Test

In [17]:
group_1 = resample_data.loc[resample_data["Approach"] == "Phonics", "W10RR"].values
group_2 = resample_data.loc[resample_data["Approach"] == "Whole-Word", "W10RR"].values

u_stat, p_value = stats.mannwhitneyu(group_1, group_2, alternative="two-sided")

pooled = np.concatenate([group_1, group_2])
ranks = stats.rankdata(pooled)
observed = ranks[:len(group_1)].sum()
null = np.array([rng.permutation(ranks)[:len(group_1)].sum() for _ in range(n_permutation)])
p_permutation = np.mean(np.abs(null - null.mean()) >= abs(observed - null.mean()))

print("Phonics:", len(group_1), "| median:", np.median(group_1))
print("Whole-Word:", len(group_2), "| median:", np.median(group_2))
print("U:", u_stat, "| rank sum:", observed)
print("p-value (asymptotic):", p_value)
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Phonics: 645 | median: 156.0
Whole-Word: 656 | median: 146.0
U: 232301.5 | rank sum: 440636.5
p-value (asymptotic): 0.0022042750573511337
p-value (permutation): 0.005
Decision: reject H0


#### Wilcoxon Signed-Rank Test For Paired t-Test

In [18]:
differences = (resample_data["W10RR"] - resample_data["W01RR"]).values
differences = differences[differences != 0]

w_stat, p_value = stats.wilcoxon(differences)

ranks = stats.rankdata(np.abs(differences))
observed = ranks[differences > 0].sum()
signs = rng.choice([-1, 1], size=(n_permutation, len(ranks)))
null = (ranks * (signs > 0)).sum(axis=1)
p_permutation = np.mean(np.abs(null - null.mean()) >= abs(observed - null.mean()))

print("Pairs:", len(differences), "| median difference:", np.median(differences))
print("W:", w_stat, "| positive rank sum:", observed)
print("p-value (asymptotic):", p_value)
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Pairs: 2000 | median difference: 27.0
W: 0.0 | positive rank sum: 2001000.0
p-value (asymptotic): 0.0
p-value (permutation): 0.0
Decision: reject H0


#### Kruskal–Wallis H Test For One-Way ANOVA Test

In [19]:
codes = resample_data["Severity"].cat.codes.values
values = resample_data["W10RR"].values
levels = np.unique(codes)

h_stat, p_value = stats.kruskal(*[values[codes == c] for c in levels])

null = np.empty(n_permutation)
for i in range(n_permutation):
    shuffled = rng.permutation(codes)
    null[i] = stats.kruskal(*[values[shuffled == c] for c in levels]).statistic
p_permutation = np.mean(null >= h_stat)

print(resample_data.groupby("Severity", observed=True)["W10RR"].median())
print("H:", round(h_stat, 4), "| df:", len(levels) - 1)
print("p-value (asymptotic):", p_value)
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Severity
Very Mild      160.0
Mild           164.0
Moderate       154.0
Severe         141.5
Very Severe    137.5
Name: W10RR, dtype: float64
H: 44.0692 | df: 4
p-value (asymptotic): 6.206896909075961e-09
p-value (permutation): 0.0
Decision: reject H0


#### Scheirer–Ray–Hare (SRH) Test For Two-Way ANOVA Test

In [20]:
srh = resample_data.copy()
srh["Rank"] = stats.rankdata(srh["W10RR"])
mean_square_total = srh["Rank"].var(ddof=1)

table = anova_lm(ols("Rank ~ C(Severity) * C(Approach)", data=srh).fit(), typ=2)
print(table)

for effect in ["C(Severity)", "C(Approach)", "C(Severity):C(Approach)"]:
    h_stat = table.loc[effect, "sum_sq"] / mean_square_total
    df_effect = int(table.loc[effect, "df"])
    p_value = 1 - stats.chi2.cdf(h_stat, df_effect)
    print(effect, "| H:", round(h_stat, 4), "| df:", df_effect,
          "| p-value:", p_value, "|", decide(p_value))

                               sum_sq      df             F    PR(>F)
C(Severity)              3.464005e+03     4.0  2.647249e-03  0.999812
C(Approach)             -7.073344e-07     2.0 -1.081113e-12  1.000000
C(Severity):C(Approach)  5.193037e+05     8.0  1.984302e-01  0.656039
Residual                 6.513209e+08  1991.0           NaN       NaN
C(Severity) | H: 0.0104 | df: 4 | p-value: 0.9999865593301013 | fail to reject H0
C(Approach) | H: -0.0 | df: 2 | p-value: 1.0 | fail to reject H0
C(Severity):C(Approach) | H: 1.5572 | df: 8 | p-value: 0.9917163516678285 | fail to reject H0


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 4, but rank is 3
  warnings.warn('covariance of constraints does not have full '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 8, but rank is 1
  warnings.warn('covariance of constraints does not have full '


#### Friedman Test For One-Way Repeated Measures ANOVA Test

In [21]:
matrix = resample_data[["W01RR", "W05RR", "W10RR"]].values

q_stat, p_value = stats.friedmanchisquare(*matrix.T)

null = np.empty(n_permutation)
for i in range(n_permutation):
    shuffled = rng.permuted(matrix, axis=1)
    null[i] = stats.friedmanchisquare(*shuffled.T).statistic
p_permutation = np.mean(null >= q_stat)

print(pd.DataFrame(matrix, columns=["W01RR", "W05RR", "W10RR"]).median())
print("Q:", round(q_stat, 4), "| df:", matrix.shape[1] - 1)
print("p-value (asymptotic):", p_value)
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

W01RR    127.0
W05RR    138.0
W10RR    153.0
dtype: float64
Q: 4000.0 | df: 2
p-value (asymptotic): 0.0
p-value (permutation): 0.0
Decision: reject H0


#### Aligned Rank Transform Test For Two-Way Repeated Measures ANOVA Test

In [22]:
subjects = resample_data.sample(n=400, random_state=42)
frames = []
for week in ["W01", "W10"]:
    frames.append(pd.DataFrame({
        "ID": subjects["ID"].values,
        "Week": week,
        "Approach": subjects["Approach"].values,
        "Score": subjects[week + "RR"].values,
    }))
art = pd.concat(frames, ignore_index=True)

grand_mean = art["Score"].mean()
cell_mean = art.groupby(["Week", "Approach"])["Score"].transform("mean")
week_mean = art.groupby("Week")["Score"].transform("mean")
approach_mean = art.groupby("Approach")["Score"].transform("mean")

art["Aligned_Week"] = art["Score"] - cell_mean + (week_mean - grand_mean)
art["Aligned_Approach"] = art["Score"] - cell_mean + (approach_mean - grand_mean)
art["Aligned_Interaction"] = art["Score"] - week_mean - approach_mean + grand_mean

for effect, column in [("C(Week)", "Aligned_Week"),
                       ("C(Approach)", "Aligned_Approach"),
                       ("C(Week):C(Approach)", "Aligned_Interaction")]:
    art["Ranked"] = stats.rankdata(art[column])
    table = anova_lm(ols("Ranked ~ C(Week) * C(Approach)", data=art).fit(), typ=2)
    f_stat = table.loc[effect, "F"]
    p_value = 1 - stats.f.cdf(f_stat, table.loc[effect, "df"], table.loc["Residual", "df"])
    print(effect, "| F:", round(f_stat, 4), "| df:", int(table.loc[effect, "df"]),
          int(table.loc["Residual", "df"]), "| p-value:", p_value, "|", decide(p_value))

C(Week) | F: 73.708 | df: 1 794 | p-value: 1.1102230246251565e-16 | reject H0
C(Approach) | F: 2.2158 | df: 2 794 | p-value: 0.10973919272730581 | fail to reject H0
C(Week):C(Approach) | F: 0.4441 | df: 2 794 | p-value: 0.6415627812129485 | fail to reject H0


#### One-Way PERMANOVA Test For One-Way MANOVA Test

In [23]:
response = distance_data[["W10RR", "W10RA", "W10RC"]].values.astype(float)
codes = distance_data["Severity"].cat.codes.values
n = len(response)

distance = np.linalg.norm(response[:, None, :] - response[None, :, :], axis=2)
squared = distance ** 2
total_sum_of_squares = squared.sum() / (2 * n)

def within_sum_of_squares(labels):
    total = 0.0
    for label in np.unique(labels):
        index = np.where(labels == label)[0]
        total += squared[np.ix_(index, index)].sum() / (2 * len(index))
    return total

def pseudo_f(labels):
    groups = len(np.unique(labels))
    within = within_sum_of_squares(labels)
    return ((total_sum_of_squares - within) / (groups - 1)) / (within / (n - groups))

observed = pseudo_f(codes)
null = np.array([pseudo_f(rng.permutation(codes)) for _ in range(n_permutation)])
p_permutation = (np.sum(null >= observed) + 1) / (n_permutation + 1)

print("Observations:", n, "| Groups:", len(np.unique(codes)))
print("Total sum of squares:", round(total_sum_of_squares, 3))
print("Pseudo-F:", round(observed, 4))
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Observations: 300 | Groups: 5
Total sum of squares: 1338552.553
Pseudo-F: 2.6899
p-value (permutation): 0.029970029970029972
Decision: reject H0


#### Two-Way PERMANOVA Test For Two-Way MANOVA Test

In [24]:
codes_a = distance_data["Severity"].cat.codes.values
codes_b = pd.Categorical(distance_data["Approach"]).codes
codes_cell = codes_a * 10 + codes_b

levels_a = len(np.unique(codes_a))
levels_b = len(np.unique(codes_b))
df_a = levels_a - 1
df_b = levels_b - 1
df_ab = df_a * df_b
df_residual = n - levels_a * levels_b

def two_way_pseudo_f(labels_a, labels_b):
    residual = within_sum_of_squares(labels_a * 10 + labels_b)
    ss_a = total_sum_of_squares - within_sum_of_squares(labels_a)
    ss_b = total_sum_of_squares - within_sum_of_squares(labels_b)
    ss_ab = (total_sum_of_squares - residual) - ss_a - ss_b
    mean_residual = residual / df_residual
    return ((ss_a / df_a) / mean_residual,
            (ss_b / df_b) / mean_residual,
            (ss_ab / df_ab) / mean_residual)

observed = two_way_pseudo_f(codes_a, codes_b)
null = np.array([two_way_pseudo_f(rng.permutation(codes_a), rng.permutation(codes_b))
                 for _ in range(n_permutation)])

for i, effect in enumerate(["Severity", "Approach", "Severity:Approach"]):
    p_permutation = (np.sum(null[:, i] >= observed[i]) + 1) / (n_permutation + 1)
    print(effect, "| Pseudo-F:", round(observed[i], 4),
          "| p-value (permutation):", p_permutation, "|", decide(p_permutation))

Severity | Pseudo-F: 2.6371 | p-value (permutation): 0.025974025974025976 | reject H0
Approach | Pseudo-F: 0.2758 | p-value (permutation): 0.7502497502497503 | fail to reject H0
Severity:Approach | Pseudo-F: 0.4563 | p-value (permutation): 0.8301698301698301 | fail to reject H0


#### One-Way Repeated Measures PERMANOVA Test For One-Way Repeated Measures MANOVA Test

In [25]:
weeks = ["W01", "W05", "W10"]
subject_count = len(distance_data)
stacked = np.vstack([distance_data[[w + "RR", w + "RA", w + "RC"]].values for w in weeks])
stacked = stacked.astype(float)
week_codes = np.repeat(np.arange(len(weeks)), subject_count)
total = len(stacked)

squared_rm = np.linalg.norm(stacked[:, None, :] - stacked[None, :, :], axis=2) ** 2
total_rm = squared_rm.sum() / (2 * total)

def within_rm(labels):
    value = 0.0
    for label in np.unique(labels):
        index = np.where(labels == label)[0]
        value += squared_rm[np.ix_(index, index)].sum() / (2 * len(index))
    return value

def pseudo_f_rm(labels):
    groups = len(np.unique(labels))
    within = within_rm(labels)
    return ((total_rm - within) / (groups - 1)) / (within / (total - groups))

observed = pseudo_f_rm(week_codes)
null = np.empty(n_permutation)
for i in range(n_permutation):
    shuffled = np.argsort(rng.random((len(weeks), subject_count)), axis=0).ravel()
    null[i] = pseudo_f_rm(shuffled)
p_permutation = (np.sum(null >= observed) + 1) / (n_permutation + 1)

print("Subjects:", subject_count, "| Repeated levels:", len(weeks))
print("Pseudo-F:", round(observed, 4))
print("p-value (within-subject permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Subjects: 300 | Repeated levels: 3
Pseudo-F: 38.6076
p-value (within-subject permutation): 0.000999000999000999
Decision: reject H0


#### Two-Way Repeated Measures PERMANOVA Test For Two-Way Repeated Measures MANOVA Test

In [26]:
approach_codes = np.tile(pd.Categorical(distance_data["Approach"]).codes, len(weeks))
levels_week = len(np.unique(week_codes))
levels_approach = len(np.unique(approach_codes))
df_week = levels_week - 1
df_approach = levels_approach - 1
df_interaction = df_week * df_approach
df_error = total - levels_week * levels_approach

def two_way_pseudo_f_rm(labels_week, labels_approach):
    residual = within_rm(labels_week * 10 + labels_approach)
    ss_week = total_rm - within_rm(labels_week)
    ss_approach = total_rm - within_rm(labels_approach)
    ss_interaction = (total_rm - residual) - ss_week - ss_approach
    mean_residual = residual / df_error
    return ((ss_week / df_week) / mean_residual,
            (ss_approach / df_approach) / mean_residual,
            (ss_interaction / df_interaction) / mean_residual)

observed = two_way_pseudo_f_rm(week_codes, approach_codes)
null = np.empty((n_permutation, 3))
for i in range(n_permutation):
    shuffled_week = np.argsort(rng.random((levels_week, subject_count)), axis=0).ravel()
    shuffled_subject = np.tile(rng.permutation(approach_codes[:subject_count]), levels_week)
    null[i] = two_way_pseudo_f_rm(shuffled_week, shuffled_subject)

for i, effect in enumerate(["Week", "Approach", "Week:Approach"]):
    p_permutation = (np.sum(null[:, i] >= observed[i]) + 1) / (n_permutation + 1)
    print(effect, "| Pseudo-F:", round(observed[i], 4),
          "| p-value (permutation):", p_permutation, "|", decide(p_permutation))

Week | Pseudo-F: 38.4875 | p-value (permutation): 0.000999000999000999 | reject H0
Approach | Pseudo-F: 1.293 | p-value (permutation): 0.5684315684315684 | fail to reject H0
Week:Approach | Pseudo-F: 0.1561 | p-value (permutation): 0.2817182817182817 | fail to reject H0


### Non-Parametric Test For Nominal Categorical Inter-Group

#### Cramer’s V With Chi-Square Homogeneity Test

In [27]:
gain = data["W10RR"] - data["W01RR"]
data["Response"] = np.where(gain > gain.median(), "High Gain", "Low Gain")

table = pd.crosstab(data["Approach"], data["Response"])
chi_square, p_value, degrees, expected = stats.chi2_contingency(table)
v = np.sqrt(chi_square / (table.values.sum() * (min(table.shape) - 1)))

print(table)
print("Chi-square:", round(chi_square, 4), "| df:", degrees)
print("Cramer's V:", round(v, 4))
print("Relationship strength:", interpret(v, [0.1, 0.3, 0.5],
                                          ["negligible", "weak", "moderate", "strong"]))
print("p-value (upper-tailed):", p_value)
print("Decision:", decide(p_value))

Response       High Gain  Low Gain
Approach                          
Multi-Sensory      16551     20222
Phonics            18595     13102
Whole-Word          9574     21956
Chi-square: 5122.861 | df: 2
Cramer's V: 0.2263
Relationship strength: weak
p-value (upper-tailed): 0.0
Decision: reject H0


## INTER-VARIABLE ANALYSIS

### Parametric Test For Continuous Numerical Inter-Variables

#### Pearson Correlation Test

In [28]:
x = data["RR0"]
y = data["RC0"]

r, p_value = stats.pearsonr(x, y)
n = len(x)
t_stat = r * np.sqrt((n - 2) / (1 - r ** 2))
p_manual = 2 * (1 - stats.t.cdf(abs(t_stat), n - 2))

print("Pairs:", n)
print("Pearson r:", round(r, 4))
print("Direction:", "positive" if r > 0 else "negative")
print("Relationship:", interpret(r, [0.3, 0.7, 1.0],
                                 ["weak", "moderate", "strong", "perfect"]))
print("t:", round(t_stat, 4), "| df:", n - 2)
print("p-value (two-tailed):", p_value)
print("p-value (CDF):", p_manual)
print("Decision:", decide(p_value))

Pairs: 100000
Pearson r: 0.9925
Direction: positive
Relationship: strong
t: 2561.4779 | df: 99998
p-value (two-tailed): 0.0
p-value (CDF): 0.0
Decision: reject H0


### Non-Parametric Test For Numerical & Ordinal Categorical Inter-Variables

#### Spearman Rank Correlation Test

In [29]:
x = resample_data["RR0"].values
y = resample_data["Level"].values

rho, p_value = stats.spearmanr(x, y)

null = np.array([stats.spearmanr(x, rng.permutation(y)).statistic
                 for _ in range(n_permutation)])
p_permutation = (np.sum(np.abs(null) >= abs(rho)) + 1) / (n_permutation + 1)

print("Pairs:", len(x))
print("Spearman rho:", round(rho, 4))
print("Direction:", "positive" if rho > 0 else "negative")
print("Relationship:", interpret(rho, [0.3, 0.7, 1.0],
                                 ["weak", "moderate", "strong", "perfect"]))
print("p-value (asymptotic):", p_value)
print("p-value (permutation):", p_permutation)
print("Decision:", decide(p_permutation))

Pairs: 2000
Spearman rho: 0.9496
Direction: positive
Relationship: strong
p-value (asymptotic): 0.0
p-value (permutation): 0.000999000999000999
Decision: reject H0


### Non-Parametric Test For Nominal Categorical Inter-Variables

#### Cramer’s V With Chi-Square Independence Test

In [30]:
table = pd.crosstab(data["Approach"], data["Severity"])
chi_square, p_value, degrees, expected = stats.chi2_contingency(table)
v = np.sqrt(chi_square / (table.values.sum() * (min(table.shape) - 1)))

print(table)
print("Chi-square:", round(chi_square, 4), "| df:", degrees)
print("Cramer's V:", round(v, 4))
print("Relationship strength:", interpret(v, [0.1, 0.3, 0.5],
                                          ["negligible", "weak", "moderate", "strong"]))
print("p-value (upper-tailed):", p_value)
print("Decision:", decide(p_value))

Severity       Very Mild  Mild  Moderate  Severe  Very Severe
Approach                                                     
Multi-Sensory          0  6940     22769    7064            0
Phonics             2000  7060     22637       0            0
Whole-Word             0     0     22594    6936         2000
Chi-square: 22148.1795 | df: 8
Cramer's V: 0.3328
Relationship strength: moderate
p-value (upper-tailed): 0.0
Decision: reject H0
